Download and Unzip the Dataset

In [1]:
#Download the full Traditional Chinese Hand Writing Dataset
!git clone https://github.com/chenkenanalytic/handwritting_data_all.git

#Unzip
!cat handwritting_data_all/all_data.zip* > all_data.zip
!unzip -q -O big5 all_data.zip -d dataset_raw

#Check
!ls dataset_raw/cleaned_data | wc -l


Cloning into 'handwritting_data_all'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 29 (delta 4), reused 11 (delta 3), pack-reused 17 (from 1)
Receiving objects: 100% (29/29), 926.83 MiB | 18.17 MiB/s, done.
Resolving deltas: 100% (4/4), done.
Updating files: 100% (15/15), done.
13065


Data Spliting (40 for trainning and 10 for testing) and Augmenting dataset to 200 images using OpenCV

In [2]:
import os
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

RAW_DIR = Path("dataset_raw/cleaned_data")
TRAIN_DIR = Path("data_split/train")
TEST_DIR = Path("data_split/test")
IMG_SIZE = 48  # resize to 48*48

def augment_image(img):
    """using OpenCV generate 4 transform images（ 1 to 5，40 to 200）"""
    h, w = img.shape[:2]
    aug_list = []

    # 1. Rotation: -12 to +12
    angle = np.random.uniform(-12, 12)
    M_rot = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    aug_list.append(cv2.warpAffine(img, M_rot, (w, h), borderValue=255))

    # 2. Scaling: 0.85x to 1.15x
    scale = np.random.uniform(0.85, 1.15)
    M_scale = cv2.getRotationMatrix2D((w // 2, h // 2), 0, scale)
    aug_list.append(cv2.warpAffine(img, M_scale, (w, h), borderValue=255))

    # 3. Shearing: Simulated handwritten with angles
    shear = np.random.uniform(-0.15, 0.15)
    M_shear = np.float32([[1, shear, 0], [0, 1, 0]])
    aug_list.append(cv2.warpAffine(img, M_shear, (w, h), borderValue=255))

    # 4. Translation
    tx, ty = np.random.randint(-3, 4), np.random.randint(-3, 4)
    M_trans = np.float32([[1, 0, tx], [0, 1, ty]])
    aug_list.append(cv2.warpAffine(aug_list[0], M_trans, (w, h), borderValue=255))

    return aug_list

char_folders = sorted([f for f in RAW_DIR.iterdir() if f.is_dir()])

for char_dir in tqdm(char_folders, desc="processing"):
    images = sorted(list(char_dir.glob("*.png")))
    if len(images) < 41:
        continue

    train_imgs = images[:40]  # first 40 images for Training
    test_imgs = images[40:]   # last 10 images for Testing

    (TRAIN_DIR / char_dir.name).mkdir(parents=True, exist_ok=True)
    (TEST_DIR / char_dir.name).mkdir(parents=True, exist_ok=True)

    # handling Training 40 to 200
    for idx, img_path in enumerate(train_imgs):
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        cv2.imwrite(str(TRAIN_DIR / char_dir.name / f"orig_{idx}.png"), img)
        for a_idx, a_img in enumerate(augment_image(img)):
            cv2.imwrite(str(TRAIN_DIR / char_dir.name / f"aug_{idx}_{a_idx}.png"), a_img)

    # handling Testing
    for idx, img_path in enumerate(test_imgs):
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        cv2.imwrite(str(TEST_DIR / char_dir.name / f"test_{idx}.png"), img)

print("Done")

processing: 100%|██████████| 13065/13065 [21:28<00:00, 10.14it/s]

Done


Load data and train CNN model

In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models, mixed_precision

# Clear previous GPU session memory
tf.keras.backend.clear_session()

# turn on T4 GPU acceration
mixed_precision.set_global_policy('mixed_float16')

IMG_SIZE = 32
BATCH_SIZE = 1024
NUM_CLASSES = 13065

# loading data keep uint8 format，2.6 million images take around 2.6GB RAM
raw_train_ds = tf.keras.utils.image_dataset_from_directory(
    "data_split/train",
    color_mode="grayscale",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True
)

raw_test_ds = tf.keras.utils.image_dataset_from_directory(
    "data_split/test",
    color_mode="grayscale",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

# 4. store into system RAM (avoid loading hard disk delay)
AUTOTUNE = tf.data.AUTOTUNE
normalization_layer = layers.Rescaling(1./255)

train_ds_fast = (
    raw_train_ds
    .cache("train_cache.tfcache")
    .shuffle(200)
    .map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

test_ds_fast = (
    raw_test_ds
    .cache("test_cache.tfcache")
    .map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)




Found 2604400 files belonging to 13022 classes.
Found 162550 files belonging to 13022 classes.


Model A: Standard 3-Layers CNN

In [2]:
# Model A: Standard 3-Layers CNN

# Clear previous GPU session memory
tf.keras.backend.clear_session()

def build_cnn_model_a():
    model = models.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),

        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model_a = build_cnn_model_a()

# 6 epochs = about 15.6 million images
history_a = model_a.fit(
    train_ds_fast,
    validation_data=test_ds_fast,
    epochs=6
)

# Evaluate Testing Accuracy and Save Model A
test_loss, test_acc = model_a.evaluate(test_ds_fast)
model_a.save("model_a.keras")
print(f"Model A Testing Accuracy:: {test_acc * 100:.2f}%")

Epoch 1/6
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 1198s 430ms/step - accuracy: 0.2362 - loss: 4.4408 - val_accuracy: 0.0220 - val_loss: 15.4297
Epoch 2/6
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 142s 55ms/step - accuracy: 0.5565 - loss: 1.8405 - val_accuracy: 0.0146 - val_loss: 19.6351
Epoch 3/6
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 145s 56ms/step - accuracy: 0.6578 - loss: 1.3330 - val_accuracy: 0.0222 - val_loss: 22.5879
Epoch 4/6
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 145s 56ms/step - accuracy: 0.7121 - loss: 1.0720 - val_accuracy: 0.0225 - val_loss: 23.6091
Epoch 5/6
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 144s 56ms/step - accuracy: 0.7481 - loss: 0.9098 - val_accuracy: 0.0270 - val_loss: 26.0395
Epoch 6/6
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 145s 56ms/step - accuracy: 0.7733 - loss: 0.8009 - val_accuracy: 0.3508 - val_loss: 5.2102
159/159 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.3508 - loss: 5.2102
Model A Testing Accuracy:: 35.08%


Model B Shallow 2-Layers CNN

In [4]:
# Clear previous GPU session memory
tf.keras.backend.clear_session()

# Model B: Shallow 2-Layers CNN (Baseline Model without BatchNormalization)
def build_cnn_model_b():
    model = models.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),

        # First Convolutional Layer
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        # Second Convolutional Layer
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        # Fully Connected Layer and Output Layer (13,065 neurons)
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [7]:
# Build and view Model B structure
model_b = build_cnn_model_b()
model_b.summary()

# Start Training Model B (6 Epochs)
history_b = model_b.fit(
    train_ds_fast,
    validation_data=test_ds_fast,
    epochs=6
)

# Evaluate Testing Accuracy and Save Model B
loss_b, acc_b = model_b.evaluate(test_ds_fast)
model_b.save("model_b_baseline.keras")
print(f"Model B Testing Accuracy: {acc_b * 100:.2f}%")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 30, 30, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 15, 15, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 13, 13, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2304)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 13065)          │     3,357,705 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,966,601 (15.13 MB)

 Trainable params: 3,966,601 (15.13 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/6
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 110s 39ms/step - accuracy: 0.0301 - loss: 7.3289 - val_accuracy: 0.0616 - val_loss: 6.1333
Epoch 2/6
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 56s 22ms/step - accuracy: 0.1208 - loss: 5.4176 - val_accuracy: 0.1376 - val_loss: 5.1072
Epoch 3/6
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 58s 21ms/step - accuracy: 0.2069 - loss: 4.4994 - val_accuracy: 0.1920 - val_loss: 4.5602
Epoch 4/6
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 58s 21ms/step - accuracy: 0.2639 - loss: 4.0185 - val_accuracy: 0.2259 - val_loss: 4.2786
Epoch 5/6
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 58s 21ms/step - accuracy: 0.3022 - loss: 3.7216 - val_accuracy: 0.2501 - val_loss: 4.1195
Epoch 6/6
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 58s 21ms/step - accuracy: 0.3301 - loss: 3.5170 - val_accuracy: 0.2647 - val_loss: 4.0060
159/159 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.2647 - loss: 4.0060
Model B Testing Accuracy: 26.47%


Model C Deep 5-Layers Double-Conv CNN with BatchNorm & Spatial Flatten

In [8]:
# Clear previous GPU session memory
tf.keras.backend.clear_session()

# Model C: Deep 5-Layers Double-Conv CNN with BatchNorm & Spatial Flatten
def build_cnn_model_c():
    model = models.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),

        # First Convolutional Block (Double Conv + BatchNorm)
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.2),

        # Second Convolutional Block (Double Conv + BatchNorm)
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),

        # Third Convolutional Block
        layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),

        # Spatial Flatten + Dense Classifier + Output Layer (13,065 neurons)
        layers.Flatten(),
        layers.Dense(1024, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [9]:
# Build and view Model C structure
model_c = build_cnn_model_c()
model_c.summary()

# Auto-reduce learning rate when validation loss stops improving
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2
)

# Start Training Model C (12 Epochs for highest accuracy)
history_c = model_c.fit(
    train_ds_fast,
    validation_data=test_ds_fast,
    epochs=12,
    callbacks=[lr_scheduler]
)

# Evaluate Testing Accuracy and Save Model C
loss_c, acc_c = model_c.evaluate(test_ds_fast)
model_c.save("model_c_best.keras")
print(f"Model C Testing Accuracy: {acc_c * 100:.2f}%")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 8, 8, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 8, 8, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1024)           │     4,195,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 13065)          │    13,391,625 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,147,785 (69.23 MB)

 Trainable params: 18,144,457 (69.22 MB)

 Non-trainable params: 3,328 (13.00 KB)

Epoch 1/12
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 700s 257ms/step - accuracy: 0.4215 - loss: 3.1421 - val_accuracy: 0.0331 - val_loss: 21.9893 - learning_rate: 0.0010
Epoch 2/12
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 603s 236ms/step - accuracy: 0.6837 - loss: 1.2832 - val_accuracy: 0.0788 - val_loss: 13.7083 - learning_rate: 0.0010
Epoch 3/12
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 603s 236ms/step - accuracy: 0.7422 - loss: 1.0029 - val_accuracy: 0.6690 - val_loss: 2.1270 - learning_rate: 0.0010
Epoch 4/12
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 603s 236ms/step - accuracy: 0.7759 - loss: 0.8489 - val_accuracy: 0.0329 - val_loss: 58.7666 - learning_rate: 0.0010
Epoch 5/12
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 602s 235ms/step - accuracy: 0.7986 - loss: 0.7473 - val_accuracy: 0.2859 - val_loss: 8.2413 - learning_rate: 0.0010
Epoch 6/12
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 601s 235ms/step - accuracy: 0.8440 - loss: 0.5599 - val_accuracy: 0.5098 - val_loss: 2.9972 - learning_rate: 5.0000e-04
Epoch 7/12
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 602

In [10]:
#Recompile Model C with a very low learning rate (1e-5) to stabilize BatchNormalization statistics
model_c.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

#Save only the best model checkpoint based on validation accuracy to avoid late-epoch fluctuations
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "model_c_best.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

#Fine-tune for 2 epochs to recover and lock in peak accuracy (~79%+)
history_c_tune = model_c.fit(
    train_ds_fast,
    validation_data=test_ds_fast,
    epochs=2,
    callbacks=[checkpoint]
)

#Load the best saved checkpoint and evaluate final testing accuracy
best_model_c = tf.keras.models.load_model("model_c_best.keras")
loss_c, acc_c = best_model_c.evaluate(test_ds_fast)
print(f"Model C Best Testing Accuracy: {acc_c * 100:.2f}%")

Epoch 1/2
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 0s 235ms/step - accuracy: 0.8970 - loss: 0.3464
Epoch 1: val_accuracy improved from None to 0.85523, saving model to model_c_best.keras

Epoch 1: finished saving model to model_c_best.keras
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 633s 242ms/step - accuracy: 0.9031 - loss: 0.3242 - val_accuracy: 0.8552 - val_loss: 0.5958
Epoch 2/2
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 0s 231ms/step - accuracy: 0.9031 - loss: 0.3257
Epoch 2: val_accuracy improved from 0.85523 to 0.85923, saving model to model_c_best.keras

Epoch 2: finished saving model to model_c_best.keras
2544/2544 ━━━━━━━━━━━━━━━━━━━━ 608s 238ms/step - accuracy: 0.9076 - loss: 0.3085 - val_accuracy: 0.8592 - val_loss: 0.5692
159/159 ━━━━━━━━━━━━━━━━━━━━ 12s 66ms/step - accuracy: 0.8592 - loss: 0.5692
Model C Best Testing Accuracy: 85.92%
